# HYPERPARAMETER TUNING FOR EVERY STOCK AND FOR EVERY TIMEFRAME

In this notebook, we are specially focusing on finding the best parameters for every possible combination of stock symbol and timeframe. as we cant fit all stocks on a single result of hyperparameter tuning. 



Note: This provides reasonable params as we have artificially injected the anomalies. These are not the best optimal params as this would need labelled data.

## Imports

In [7]:
import sys
from pathlib import Path
import hashlib
from datetime import date
from dateutil.relativedelta import relativedelta
import warnings
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.neighbors import NearestNeighbors

sys.path.append(str(Path().resolve().parent))

from src.utils.timeframes import VALID_TIMEFRAMES, WINDOW_MAP
from src.utils.load import load_config
from src.utils.paths import HYPERPARAMS, CONFIG
from src.models.isolation_forest import IsolationForest
from src.models.dbscan import DBSCAN
from src.components.data_loader import DataLoader
from src.components.preprocessing import Preprocessor
from src.components.scaling import FeatureScaler
from src.utils.io import get_symbols
from itertools import product

warnings.filterwarnings("ignore")

SEED = 42
RUN_END_DATE = date(2026, 5, 30)


def stable_seed(*parts):
    """Build a deterministic seed from a shared base seed plus run labels."""
    payload = ":".join([str(SEED), *map(str, parts)])
    return int(hashlib.sha256(payload.encode("utf-8")).hexdigest()[:16], 16) % (2**32)


def make_rng(*parts):
    return np.random.default_rng(stable_seed(*parts))


def _event_windows(labels):
    """Return contiguous anomaly windows as inclusive (start, end) pairs."""
    windows = []
    start = None

    for idx, value in enumerate(labels):
        if value == 1 and start is None:
            start = idx
        elif value != 1 and start is not None:
            windows.append((start, idx - 1))
            start = None

    if start is not None:
        windows.append((start, len(labels) - 1))

    return windows


def build_validation_slices(length, max_slices=3, min_slice_size=80):
    """Build contiguous slices for stability-aware tuning."""
    if length <= min_slice_size:
        return [(0, length)]

    slice_size = max(min_slice_size, length // max_slices)
    slices = []
    start = 0

    while start < length:
        end = min(length, start + slice_size)
        if end - start >= min_slice_size:
            slices.append((start, end))
        start = end

    if not slices:
        slices = [(0, length)]

    return slices[:max_slices]


def compute_row_metrics(y_true, y_pred):
    """Optional diagnostic metrics at row level (not used for tuning)."""
    y_true_binary = (np.asarray(y_true) == 1).astype(int)
    y_pred_binary = (np.asarray(y_pred) == 1).astype(int)
    return {
        "precision": precision_score(y_true_binary, y_pred_binary, zero_division=0),
        "recall": recall_score(y_true_binary, y_pred_binary, zero_division=0),
        "f1_score": f1_score(y_true_binary, y_pred_binary, zero_division=0),
    }


def summarize_predictions(y_true, y_pred):
    """Event-only scoring used for hyperparameter tuning."""
    y_true_binary = (np.asarray(y_true) == 1).astype(int)
    y_pred_binary = (np.asarray(y_pred) == 1).astype(int)
    event_scores = score_event_detection(y_true_binary, y_pred_binary)

    return {
        "event_precision": event_scores["event_precision"],
        "event_recall": event_scores["event_recall"],
        "event_f1": event_scores["event_f1"],
        "avg_detection_lag": event_scores["avg_detection_lag"],
        "detected_events": event_scores["detected_events"],
        "true_events": event_scores["true_events"],
        "pred_events": event_scores["pred_events"],
        "score": event_scores["event_f1"],
    }


def score_event_detection(y_true, y_pred):
    """Score predictions at the anomaly-event level."""
    true_windows = _event_windows(y_true)
    pred_windows = _event_windows(y_pred)

    if not true_windows:
        return {
            "event_precision": 0.0,
            "event_recall": 0.0,
            "event_f1": 0.0,
            "avg_detection_lag": None,
            "detected_events": 0,
            "true_events": 0,
            "pred_events": len(pred_windows),
        }

    matched_true = set()
    detection_lags = []
    matched_predictions = 0

    for pred_start, pred_end in pred_windows:
        overlap_index = None
        for true_index, (true_start, true_end) in enumerate(true_windows):
            if pred_end >= true_start and pred_start <= true_end:
                overlap_index = true_index
                break

        if overlap_index is not None and overlap_index not in matched_true:
            matched_true.add(overlap_index)
            matched_predictions += 1
            true_start = true_windows[overlap_index][0]
            detection_lags.append(max(0, pred_start - true_start))

    event_precision = matched_predictions / len(pred_windows) if pred_windows else 0.0
    event_recall = len(matched_true) / len(true_windows) if true_windows else 0.0
    if event_precision + event_recall:
        event_f1 = 2 * event_precision * event_recall / (event_precision + event_recall)
    else:
        event_f1 = 0.0

    return {
        "event_precision": event_precision,
        "event_recall": event_recall,
        "event_f1": event_f1,
        "avg_detection_lag": float(np.mean(detection_lags)) if detection_lags else None,
        "detected_events": len(matched_true),
        "true_events": len(true_windows),
        "pred_events": len(pred_windows),
    }


## Inject Anomalies for Evaluation

| Anomaly Type      | Impact on 4D Space                                                |
| ----------------- | ----------------------------------------------------------------- |
| pump/dump         | close ↑, volume → (basic)                                         |
| volume_spike      | volume ↑↑, returns/volatility normal                              |
| inverse_volume    | volume ↓ BUT close ↑↑ (breaks correlation)                        |
| volatility_crush  | volatility ↓↓ (opposite of expected)                              |
| gap_open          | returns ↑↑ on single point (isolated)                             |
| wick_spike        | volatility ↑↑ but returns ≈ 0 (decouples returns from volatility) |
| correlation_break | volume ↑↑↑ but returns ≈ 0 (strongest break)                      |


In [8]:
def inject_anomalies(df, features, rng=None):
    """Inject synthetic anomalies with a deterministic random generator."""
    rng = rng or np.random.default_rng(SEED)

    fraction = 0.02
    price_range = (0.05, 0.40)
    vol_multiplier = (4, 12)
    window_size = (3, 8)

    df_injected = df.copy()
    df_injected["True_Anomaly"] = 0

    n = len(df_injected)
    expected_window_length = (window_size[0] + window_size[1]) / 2
    k = max(1, int((n * fraction) / expected_window_length))
    idxs = rng.choice(range(n - max(window_size)), size=k, replace=False)

    for start_idx in idxs:
        duration = int(rng.integers(*window_size))
        anomaly_type = rng.choice(
            ["pump", "dump", "volume_spike", "inverse_volume", "volatility_crush"],
            p=[0.25, 0.25, 0.20, 0.15, 0.15],
        )

        end_idx = min(n, start_idx + duration)
        for i in range(start_idx, end_idx):
            df_injected.iloc[i, df_injected.columns.get_loc("True_Anomaly")] = 1

            if anomaly_type == "pump":
                factor = rng.uniform(*price_range) + rng.normal(0, 0.003)
                for col in ["open", "high", "low", "close"]:
                    col_idx = df_injected.columns.get_loc(col)
                    df_injected.iloc[i, col_idx] *= 1 + factor

            elif anomaly_type == "dump":
                factor = -(rng.uniform(*price_range)) + rng.normal(0, 0.003)
                for col in ["open", "high", "low", "close"]:
                    col_idx = df_injected.columns.get_loc(col)
                    df_injected.iloc[i, col_idx] *= 1 + factor

            elif anomaly_type == "volume_spike":
                col_idx = df_injected.columns.get_loc("volume")
                df_injected.iloc[i, col_idx] *= rng.uniform(*vol_multiplier)

            elif anomaly_type == "inverse_volume":
                col_idx = df_injected.columns.get_loc("volume")
                df_injected.iloc[i, col_idx] *= rng.uniform(0.05, 0.2)
                factor = rng.uniform(0.10, 0.35) * rng.choice([-1, 1])
                for col in ["open", "high", "low", "close"]:
                    col_idx = df_injected.columns.get_loc(col)
                    df_injected.iloc[i, col_idx] *= 1 + factor

            elif anomaly_type == "volatility_crush":
                close_value = df_injected.iloc[i]["close"]
                epsilon = rng.uniform(0.00005, 0.0005)
                for col in ["open", "high", "low", "close"]:
                    col_idx = df_injected.columns.get_loc(col)
                    df_injected.iloc[i, col_idx] = close_value * (1 + rng.uniform(-epsilon, epsilon))
                col_idx = df_injected.columns.get_loc("volume")
                df_injected.iloc[i, col_idx] *= rng.uniform(0.3, 0.6)

    df_injected["True_Anomaly"] = df_injected["True_Anomaly"].astype(int)
    df_injected["returns"] = df_injected["close"].pct_change()
    df_injected["volatility"] = df_injected["returns"].rolling(window=20).std()
    df_injected["SMA_20"] = df_injected["close"].rolling(window=20).mean()
    df_injected["Upper_BB"] = df_injected["SMA_20"] + (df_injected["close"].rolling(window=20).std() * 2)
    df_injected["Lower_BB"] = df_injected["SMA_20"] - (df_injected["close"].rolling(window=20).std() * 2)
    df_injected["bb_width"] = (df_injected["Upper_BB"] - df_injected["Lower_BB"]) / df_injected["close"]

    df_injected.replace([np.inf, -np.inf], np.nan, inplace=True)
    df_injected.dropna(subset=features, inplace=True)

    return df_injected

## Tuning Functions for different models

In [9]:
from sklearn.neighbors import NearestNeighbors


TUNING_PROFILE = "balanced"
TUNING_FALLBACK_THRESHOLD = 0.6
TUNING_GRID_PROFILES = {
    "fast": {
        "if": {"n_estimators": [100], "contamination": [0.02, 0.03, 0.04]},
        "dbscan": {"min_pts": [5], "eps_percentiles": [92, 95, 97]},
    },
    "balanced": {
        "if": {"n_estimators": [100, 200], "contamination": [0.02, 0.03, 0.04]},
        "dbscan": {"min_pts": [5, 8], "eps_percentiles": [92, 95, 97, 99]},
    },
    "wide": {
        "if": {"n_estimators": [100, 200, 300, 500], "contamination": [0.005, 0.01, 0.015, 0.02, 0.03, 0.04, 0.05]},
        "dbscan": {"min_pts": [4, 5, 8, 12], "eps_percentiles": [80, 85, 88, 90, 92, 95, 97, 99]},
    },
}


def get_tuning_profile(name):
    return TUNING_GRID_PROFILES.get(name, TUNING_GRID_PROFILES["balanced"])


def smooth_predictions(preds, min_event_len=2, max_gap=1):
    """Reduce noisy event fragmentation in binary anomaly labels."""
    preds = np.asarray(preds).astype(int).copy()
    n = len(preds)

    if max_gap > 0:
        i = 0
        while i < n:
            if preds[i] == 1:
                j = i + 1
                while j < n and preds[j] == 1:
                    j += 1
                k = j
                while k < n and preds[k] == 0:
                    k += 1
                gap = k - j
                if k < n and 0 < gap <= max_gap:
                    preds[j:k] = 1
                i = k
            else:
                i += 1

    if min_event_len > 1:
        i = 0
        while i < n:
            if preds[i] == 1:
                j = i + 1
                while j < n and preds[j] == 1:
                    j += 1
                if (j - i) < min_event_len:
                    preds[i:j] = 0
                i = j
            else:
                i += 1

    return preds


def quick_tune_isolation_forest(X, y_true, verbose=True, max_slices=3, profile=TUNING_PROFILE):
    results = []
    slices = build_validation_slices(len(X), max_slices=max_slices)
    profile_cfg = get_tuning_profile(profile)["if"]
    combinations = list(product(profile_cfg["n_estimators"], profile_cfg["contamination"]))

    if verbose:
        print(f"Testing {len(combinations)} Isolation Forest combinations across {len(slices)} slices (profile={profile})...")

    for n_est, contam in combinations:
        try:
            slice_metrics = []

            for start, end in slices:
                X_slice = X[start:end]
                y_slice = y_true[start:end]

                if len(X_slice) < 10:
                    continue

                model = IsolationForest(n_trees=n_est, contamination=contam, random_state=42)
                predictions = model.fit_predict(X_slice)
                preds = (predictions == -1).astype(int)
                preds = smooth_predictions(preds, min_event_len=2, max_gap=1)
                slice_metrics.append(summarize_predictions(y_slice, preds))

            if not slice_metrics:
                continue

            metrics_df = pd.DataFrame(slice_metrics)
            results.append(
                {
                    "n_estimators": n_est,
                    "contamination": contam,
                    "event_precision": metrics_df["event_precision"].mean(),
                    "event_recall": metrics_df["event_recall"].mean(),
                    "event_f1": metrics_df["event_f1"].mean(),
                    "avg_detection_lag": metrics_df["avg_detection_lag"].dropna().mean() if metrics_df["avg_detection_lag"].notna().any() else None,
                    "score": metrics_df["event_f1"].mean(),
                    "slices_used": len(metrics_df),
                }
            )

        except Exception as e:
            if verbose:
                print(f"  Error with n_est={n_est}, contam={contam}: {e}")

    if not results:
        return pd.DataFrame(
            {
                "n_estimators": [profile_cfg["n_estimators"][0]],
                "contamination": [profile_cfg["contamination"][0]],
                "event_precision": [0.0],
                "event_recall": [0.0],
                "event_f1": [0.0],
                "avg_detection_lag": [None],
                "score": [0.0],
                "slices_used": [0],
            }
        )

    results_df = pd.DataFrame(results)
    results_df = results_df.sort_values(["score", "event_recall", "event_precision"], ascending=False)

    if verbose:
        best = results_df.iloc[0]
        print(f"\n✓ Best Event F1: {best['event_f1']:.4f}")
        print(
            f"✓ Best Parameters: n_estimators={int(best['n_estimators'])}, "
            f"contamination={best['contamination']:.4f}, "
            f"event_precision={best['event_precision']:.4f}, "
            f"event_recall={best['event_recall']:.4f}\n"
        )

    return results_df


def quick_tune_dbscan(X, y_true, verbose=False, max_slices=3, profile=TUNING_PROFILE):
    """DBSCAN tuning with flexible min_pts + eps grids and event_f1-only ranking."""
    print("\n" + "=" * 80)
    print("DBSCAN HYPERPARAMETER ESTIMATION")
    print("=" * 80)

    n_features = X.shape[1]
    profile_cfg = get_tuning_profile(profile)["dbscan"]
    min_pts_candidates = profile_cfg["min_pts"]
    print(f"\n[INFO] Feature dimensions: {n_features}")
    print(f"[INFO] min_pts candidates: {min_pts_candidates} (profile={profile})")

    results = []
    slices = build_validation_slices(len(X), max_slices=max_slices)

    for min_pts in min_pts_candidates:
        nn = NearestNeighbors(n_neighbors=min_pts)
        nn.fit(X)
        distances, _ = nn.kneighbors(X)
        k_distances = np.sort(distances[:, -1])
        eps_candidates = np.percentile(k_distances, profile_cfg["eps_percentiles"])

        if verbose:
            print(f"\n[INFO] min_pts={min_pts}")

        for eps in eps_candidates:
            try:
                slice_metrics = []

                for start, end in slices:
                    X_slice = X[start:end]
                    y_slice = y_true[start:end]

                    if len(X_slice) < 10:
                        continue

                    model = DBSCAN(eps=eps, min_pts=min_pts)
                    labels = np.asarray(model.fit_predict(X_slice))
                    preds = (labels == -1).astype(int)
                    preds = smooth_predictions(preds, min_event_len=2, max_gap=1)
                    slice_metrics.append(summarize_predictions(y_slice, preds))

                if not slice_metrics:
                    continue

                metrics_df = pd.DataFrame(slice_metrics)
                results.append(
                    {
                        "eps": float(eps),
                        "min_pts": int(min_pts),
                        "event_precision": metrics_df["event_precision"].mean(),
                        "event_recall": metrics_df["event_recall"].mean(),
                        "event_f1": metrics_df["event_f1"].mean(),
                        "avg_detection_lag": metrics_df["avg_detection_lag"].dropna().mean() if metrics_df["avg_detection_lag"].notna().any() else None,
                        "score": metrics_df["event_f1"].mean(),
                        "slices_used": len(metrics_df),
                    }
                )
            except Exception as e:
                if verbose:
                    print(f"[ERROR] min_pts={min_pts}, eps={eps:.4f}: {str(e)[:80]}")

    if not results:
        print("\n[WARNING] No valid DBSCAN configurations found")
        return None

    results_df = pd.DataFrame(results)
    results_df = results_df.sort_values(["score", "event_recall", "event_precision"], ascending=False)

    print("\n" + "=" * 80)
    print("BEST DBSCAN CONFIGURATION")
    print("=" * 80)

    best = results_df.iloc[0]

    print(f"Best eps: {best['eps']:.6f}")
    print(f"Best min_pts: {int(best['min_pts'])}")
    print(f"Event F1: {best['event_f1']:.4f}")
    print(f"Event Precision: {best['event_precision']:.4f}")
    print(f"Event Recall: {best['event_recall']:.4f}")

    return results_df

## Visualize hyperparameters

In [10]:
# visualize the results


def plot_if_tuning(results_df):
    """Plot Isolation Forest tuning results by event_f1."""
    try:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))

        for n_est in results_df["n_estimators"].unique():
            subset = results_df[results_df["n_estimators"] == n_est].sort_values("contamination")
            axes[0].plot(
                subset["contamination"],
                subset["event_f1"],
                marker="o",
                label=f"n_est={int(n_est)}",
            )

        axes[0].set_xlabel("Contamination")
        axes[0].set_ylabel("Event F1")
        axes[0].set_title("Isolation Forest: Event F1 vs Contamination")
        axes[0].legend()
        axes[0].grid(True, alpha=0.3)

        top_10 = results_df.head(10).sort_values("event_f1")
        axes[1].barh(range(len(top_10)), top_10["event_f1"], color="steelblue")
        labels = [
            f"n_est={int(row['n_estimators'])}, cont={row['contamination']:.3f}"
            for _, row in top_10.iterrows()
        ]
        axes[1].set_yticks(range(len(top_10)))
        axes[1].set_yticklabels(labels, fontsize=9)
        axes[1].set_xlabel("Event F1")
        axes[1].set_title("Top 10 Parameter Combinations")

        plt.tight_layout()
        plt.show()
    except Exception as e:
        print(f"Could not plot Isolation Forest results: {e}")


def plot_dbscan_tuning(results_df):
    """Plot DBSCAN tuning results as event_f1 heatmap."""
    try:
        import seaborn as sns

        if len(results_df) < 2:
            print("Not enough DBSCAN results to plot")
            return

        pivot_table = results_df.pivot_table(
            index="min_pts", columns="eps", values="event_f1", aggfunc="mean"
        )

        if pivot_table.empty:
            print("Could not create pivot table for DBSCAN")
            return

        plt.figure(figsize=(8, 6))
        sns.heatmap(
            pivot_table,
            annot=True,
            fmt=".3f",
            cmap="RdYlGn",
            cbar_kws={"label": "Event F1"},
        )
        plt.title("DBSCAN: Event F1 Heatmap")
        plt.xlabel("eps")
        plt.ylabel("min_pts")
        plt.tight_layout()
        plt.show()
    except Exception as e:
        print(f"Could not plot DBSCAN results: {e}")

## Run the functions

In [11]:
def run_hyperparameter_tuning(df, features, symbol, timeframe, verbose=True, tuning_profile=TUNING_PROFILE):
    print("\n" + "=" * 60)
    print(f"HYPERPARAMETER TUNING FOR ANOMALY DETECTION for {symbol} at {timeframe}")
    print("=" * 60 + "\n")

    scaler = FeatureScaler()
    X = scaler.fit_transform(df, features)
    y_true = df["True_Anomaly"].values

    best_params = {}

    print(" TUNING ISOLATION FOREST")
    print("-" * 60)
    results_if = quick_tune_isolation_forest(X, y_true, verbose=verbose, profile=tuning_profile)

    if not results_if.empty and results_if.iloc[0]["event_f1"] < TUNING_FALLBACK_THRESHOLD and tuning_profile != "wide":
        if verbose:
            print(f"\n[INFO] Isolation Forest F1 below {TUNING_FALLBACK_THRESHOLD:.2f}; retrying with wide profile...")
        fallback_if = quick_tune_isolation_forest(X, y_true, verbose=verbose, profile="wide")
        if not fallback_if.empty and fallback_if.iloc[0]["event_f1"] > results_if.iloc[0]["event_f1"]:
            results_if = fallback_if

    best_params["isolation_forest"] = {
        "n_estimators": int(results_if.iloc[0]["n_estimators"]),
        "contamination": float(results_if.iloc[0]["contamination"]),
    }

    if verbose and len(results_if) > 0:
        print("Results:")
        cols_to_show = [
            c
            for c in [
                "n_estimators",
                "contamination",
                "event_precision",
                "event_recall",
                "event_f1",
                "score",
                "slices_used",
            ]
            if c in results_if.columns
        ]
        print(results_if[cols_to_show].head())

    print("\n TUNING DBSCAN")
    print("-" * 60)
    results_dbscan = quick_tune_dbscan(X, y_true, verbose=False, profile=tuning_profile)

    if results_dbscan is not None and not results_dbscan.empty and results_dbscan.iloc[0]["event_f1"] < TUNING_FALLBACK_THRESHOLD and tuning_profile != "wide":
        if verbose:
            print(f"\n[INFO] DBSCAN F1 below {TUNING_FALLBACK_THRESHOLD:.2f}; retrying with wide profile...")
        fallback_dbscan = quick_tune_dbscan(X, y_true, verbose=False, profile="wide")
        if fallback_dbscan is not None and not fallback_dbscan.empty and fallback_dbscan.iloc[0]["event_f1"] > results_dbscan.iloc[0]["event_f1"]:
            results_dbscan = fallback_dbscan

    best_params["dbscan"] = {
        "eps": float(results_dbscan.iloc[0]["eps"]),
        "min_pts": int(results_dbscan.iloc[0]["min_pts"]),
    }

    if verbose and len(results_dbscan) > 0:
        print("Results:")
        cols_to_show = [
            c
            for c in [
                "eps",
                "min_pts",
                "event_precision",
                "event_recall",
                "event_f1",
                "score",
                "slices_used",
            ]
            if c in results_dbscan.columns
        ]
        print(results_dbscan[cols_to_show].head())

    print("\n" + "=" * 60)
    print("SUMMARY OF BEST PARAMETERS")
    print("=" * 60)

    if len(results_if) > 0:
        best_if = results_if.iloc[0]
        print("\nISOLATION_FOREST:")
        print(f"  n_estimators: {best_if['n_estimators']}")
        print(f"  contamination: {best_if['contamination']}")
        print(f"  event_precision: {best_if.get('event_precision', 0.0):.4f}")
        print(f"  event_recall: {best_if.get('event_recall', 0.0):.4f}")
        print(f"  event_f1: {best_if.get('event_f1', 0.0):.4f}")

    if len(results_dbscan) > 0:
        best_db = results_dbscan.iloc[0]
        print("\nDBSCAN:")
        print(f"  eps: {best_db['eps']}")
        print(f"  min_pts: {best_db['min_pts']}")
        print(f"  event_precision: {best_db.get('event_precision', 0.0):.4f}")
        print(f"  event_recall: {best_db.get('event_recall', 0.0):.4f}")
        print(f"  event_f1: {best_db.get('event_f1', 0.0):.4f}")

    print("\nZ_SCORE:")
    print("  threshold: 3")

    best_params["z_score"] = {"threshold": 3}

    return best_params, {
        "if": results_if,
        "dbscan": results_dbscan,
    }


if __name__ == "__main__":
    print("This module is ready to be imported and used in your notebook.")
    print("\nUsage:")
    print("  best_params = run_hyperparameter_tuning(df, features)")

This module is ready to be imported and used in your notebook.

Usage:
  best_params = run_hyperparameter_tuning(df, features)


## Tune hyperparamters for every possible combination

In [12]:
config = load_config(CONFIG / "config.yaml")
stocks = sorted(get_symbols())
features = config["features"]

for stock in stocks:
    output_path = HYPERPARAMS / f"{stock}.json"

    print(f"Tuning hyperparameters for {stock}...")
    if output_path.exists():
        print(f"Skipping {stock}")
        continue

    best_params_stock = {}

    for timeframe in VALID_TIMEFRAMES:
        days = WINDOW_MAP[timeframe]
        loader = DataLoader()
        data = loader.load(
            symbol=stock,
            start_date=(RUN_END_DATE - relativedelta(days=days)),
            end_date=RUN_END_DATE,
        )

        preprocessor = Preprocessor()
        clean_data = preprocessor.transform(data, timeframe)

        rng = make_rng(stock, timeframe)
        df_injected = inject_anomalies(clean_data, features, rng=rng)

        print(stock, timeframe)
        print("raw:", len(data))
        print("clean:", len(clean_data))
        print(f"Injected Anomalies: {len(df_injected[df_injected['True_Anomaly'] == 1])}")

        best_params, _ = run_hyperparameter_tuning(df_injected, features, stock, timeframe)
        best_params["z_score"] = {"threshold": 3}
        best_params_stock[timeframe] = best_params

    with open(output_path, "w") as f:
        json.dump(best_params_stock, f, indent=2)
    
    


Tuning hyperparameters for ADBL...
Skipping ADBL
Tuning hyperparameters for AHPC...
Skipping AHPC
Tuning hyperparameters for AKJCL...
Skipping AKJCL
Tuning hyperparameters for AKPL...
Skipping AKPL
Tuning hyperparameters for ALICL...
Skipping ALICL
Tuning hyperparameters for API...
Skipping API
Tuning hyperparameters for BARUN...
Skipping BARUN
Tuning hyperparameters for BFC...
Skipping BFC
Tuning hyperparameters for BPCL...
Skipping BPCL
Tuning hyperparameters for CFCL...
Skipping CFCL
Tuning hyperparameters for CGH...
Skipping CGH
Tuning hyperparameters for CHCL...
Skipping CHCL
Tuning hyperparameters for CHDC...
Skipping CHDC
Tuning hyperparameters for CHL...
Skipping CHL
Tuning hyperparameters for CIT...
Skipping CIT
Tuning hyperparameters for CORBL...
Skipping CORBL
Tuning hyperparameters for CZBIL...
Skipping CZBIL
Tuning hyperparameters for DHPL...
Skipping DHPL
Tuning hyperparameters for EBL...
Skipping EBL
Tuning hyperparameters for EDBL...
Skipping EDBL
Tuning hyperparameters